In [19]:
#Import libaries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, r2_score, classification_report,
                             confusion_matrix, roc_auc_score)
try:
    from xgboost import XGBRegressor, XGBClassifier
    BACKEND = "XGBoost"
except ImportError:  # fallback only so the script still runs where xgboost is missing
    from sklearn.ensemble import GradientBoostingRegressor as XGBRegressor
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier
    BACKEND = "sklearn GradientBoosting (xgboost not installed)"
 
OUT = ""   # charts saved next to the script
rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid")

In [17]:
#Read the files 
Dataset_1 = pd.read_csv("Maharashtra_Electricity_Distribution_cleaned.csv")
Dataset_2 = pd.read_csv("POWER_Point_Daily_2025_cleaned.csv")
w = Dataset_2.copy()
w["date"] = pd.to_datetime(w["date"])
w = w.rename(columns={"solar_irradiance_kwh_m2_day": "irr", "temperature_2m_c": "temp"})
m = Dataset_1.rename(columns={"Installed Capacity in MW": "install_mw",
                      "Solar Plant Capacity in MW": "solar_mw",
                      "Land Required in Acre": "land_acre"})

In [21]:
#CORRELATION (real data)

w["month"] = w.date.dt.month
w["doy"] = w.date.dt.dayofyear
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(w[["irr", "temp", "month"]].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax[0])
ax[0].set_title("Weather correlations (real, 2025)")
sns.heatmap(m[["install_mw", "solar_mw", "land_acre"]].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax[1])
ax[1].set_title("Substation capacity correlations (real)")
plt.tight_layout(); plt.savefig(OUT + "01_correlations.png", dpi=130); plt.close()
 
monthly = w.groupby("month")[["irr", "temp"]].mean()
print("Irradiance~Temp r = %.2f" % w.irr.corr(w.temp))
print("Land/MW ratio mean=%.3f std=%.4f  (land is just 5 x MW -> redundant feature)" %
      ((m.land_acre / m.solar_mw).mean(), (m.land_acre / m.solar_mw).std()))

Irradiance~Temp r = 0.44
Land/MW ratio mean=5.000 std=0.0063  (land is just 5 x MW -> redundant feature)


In [23]:
# 3. SIMULATE PLANT (10 kW rooftop-scale)
CAP_KW = 10.0
tcell = w.temp + 4.0 * w.irr                                  # cell heats with sun
pr_temp = 1 - 0.004 * (tcell - 25)                            # ~-0.4%/degC power loss
w["expected_kwh"] = CAP_KW * w.irr * 0.80 * pr_temp           # 0.80 = system losses
 
# rain proxy: very low-irradiance days wash panels
rain = (w.irr < 3.0).to_numpy()
soil = np.zeros(len(w)); s = 0.0
for i in range(len(w)):
    s = 0.0 if rain[i] else min(s + 0.002, 0.15)             # +0.2%/dry day, capped 15%
    soil[i] = s
shade = np.zeros(len(w))                                      # random shading episodes
for st in rng.choice(len(w) - 12, 6, replace=False):
    shade[st:st + rng.integers(4, 12)] = rng.uniform(0.08, 0.18)
equip = np.zeros(len(w))                                      # equipment faults
for st in rng.choice(len(w) - 25, 3, replace=False):
    equip[st:st + rng.integers(7, 25)] = rng.uniform(0.20, 0.45)
 
loss = 1 - (1 - soil) * (1 - shade) * (1 - equip)
w["true_loss"] = loss
w["actual_kwh"] = (w.expected_kwh * (1 - loss) * rng.normal(1, 0.02, len(w))).clip(lower=0)
w["fault"] = np.select([equip > 0, shade > 0, soil > 0.08], ["equipment", "shading", "soiling"], "healthy")
w["needs_inspection"] = (loss > 0.10).astype(int)

In [25]:
#4. MODEL 1: baseline "expected output"
healthy = w[w.fault == "healthy"]
tr_idx = healthy.sample(frac=0.7, random_state=1).index
feat1 = ["irr", "temp"]
reg = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=0)
reg.fit(w.loc[tr_idx, feat1], w.loc[tr_idx, "actual_kwh"])
w["pred_kwh"] = reg.predict(w[feat1])
te_h = healthy.index.difference(tr_idx)
print("\n[Baseline regressor – %s] held-out healthy days: MAE=%.2f kWh, R2=%.3f" %
      (BACKEND, mean_absolute_error(w.loc[te_h, "actual_kwh"], w.loc[te_h, "pred_kwh"]),
       r2_score(w.loc[te_h, "actual_kwh"], w.loc[te_h, "pred_kwh"])))


[Baseline regressor – XGBoost] held-out healthy days: MAE=1.16 kWh, R2=0.982


In [26]:
#5. MODEL 2: inspection alert classifier

w["ratio"] = w.actual_kwh / w.pred_kwh.clip(lower=0.5)
w["ratio_7d"] = w.ratio.rolling(7, min_periods=1).mean()
w["ratio_slope_7d"] = w.ratio.rolling(7, min_periods=3).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0]).fillna(0)
last_rain = pd.Series(np.where(rain, np.arange(len(w)), np.nan)).ffill().fillna(0)
w["days_since_rain"] = np.arange(len(w)) - last_rain
# days_since_rain is deliberately NOT used: the simulator builds soiling from it (circular)
feat2 = ["ratio", "ratio_7d", "ratio_slope_7d", "irr", "temp"]
cut = int(len(w) * 0.7)                                       # time-ordered split, no leakage
Xtr, Xte = w.loc[:cut - 1, feat2], w.loc[cut:, feat2]
ytr, yte = w.loc[:cut - 1, "needs_inspection"], w.loc[cut:, "needs_inspection"]
clf = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0)
clf.fit(Xtr, ytr)
pred = clf.predict(Xte); prob = clf.predict_proba(Xte)[:, 1]
print("\n[Inspection classifier] test = last 30%% of year (%d days, %d need inspection)" % (len(yte), yte.sum()))
print(classification_report(yte, pred, target_names=["ok", "inspect"], digits=2))
print("ROC-AUC = %.3f" % roc_auc_score(yte, prob))
print("Simple rule 'ratio < 0.90' accuracy: %.3f  vs XGB: %.3f" %
      (((Xte.ratio < 0.9).astype(int) == yte).mean(), (pred == yte).mean()))


[Inspection classifier] test = last 30% of year (110 days, 39 need inspection)
              precision    recall  f1-score   support

          ok       0.68      0.99      0.80        71
     inspect       0.86      0.15      0.26        39

    accuracy                           0.69       110
   macro avg       0.77      0.57      0.53       110
weighted avg       0.74      0.69      0.61       110

ROC-AUC = 0.936
Simple rule 'ratio < 0.90' accuracy: 0.791  vs XGB: 0.691
